# 🗺️ ArcGIS API for Python — Fully Fixed Map & Location Notebook

### All 3 bugs fixed:
| Bug | Old (broken) | Fixed |
|---|---|---|
| Missing package | `pip install arcgis` only | `pip install arcgis-mapping` too |
| Basemap setter | `map.basemap = 'x'` | `map.basemap.basemap = 'x'` |
| Draw symbol/popup | plain `dict` | `Geometry` + `SimpleMarkerSymbolEsriSMS` pydantic model |

---

In [ ]:
# Run once if not already installed
!pip install arcgis-mapping --quiet
print('✅ arcgis-mapping ready')

## 🔌 Step 1 — Imports & Connect

In [ ]:
import arcgis
from arcgis.gis       import GIS
from arcgis.geocoding import geocode, reverse_geocode
from arcgis.geometry  import Geometry          # ✅ REQUIRED for draw()

# Symbol classes — these are pydantic models, NOT plain dicts
from arcgis.map.symbols import (
    SimpleMarkerSymbolEsriSMS,   # circle / square / diamond markers
    SimpleLineSymbolEsriSLS,     # lines / polylines
    SimpleFillSymbolEsriSFS,     # filled polygons
    PictureMarkerSymbolEsriPMS,  # custom image icons
)

# Anonymous login — no account needed
gis = GIS()
print('✅ Connected to ArcGIS:', gis)
print('✅ arcgis version     :', arcgis.__version__)

## 🗺️ Step 2 — Create a Map Widget

In [ ]:
# Pass any place name — map auto-centers there
my_map = gis.map('India')
my_map.zoom = 5
my_map.layout.height = '500px'
my_map

## 🎨 Step 3 — Change Basemap (correct syntax)

In [ ]:
# List every available basemap
print('All basemaps:')
for b in my_map.basemap.basemaps:
    print(' ', b)

In [ ]:
# ✅ Correct — note the double .basemap.basemap
my_map.basemap.basemap = 'satellite'

# Try any of these:
# my_map.basemap.basemap = 'dark-gray-vector'
# my_map.basemap.basemap = 'hybrid'
# my_map.basemap.basemap = 'topo-vector'
# my_map.basemap.basemap = 'streets-night-vector'
# my_map.basemap.basemap = 'osm'
print('✅ Basemap changed — scroll up!')

## 🔍 Step 4 — Search & Zoom to Any Location

In [ ]:
search_map = gis.map()
search_map.layout.height = '450px'
search_map

In [ ]:
# ✏️ Change this to anything: city, address, landmark
query = 'Gateway of India, Mumbai'

results = geocode(query, max_locations=1)

if results:
    r     = results[0]
    lat   = r['location']['y']
    lon   = r['location']['x']
    name  = r['address']
    score = r['score']

    print(f'📍 Found : {name}')
    print(f'   Lat   : {lat:.5f}')
    print(f'   Lon   : {lon:.5f}')
    print(f'   Score : {score}/100  (confidence)')

    # Zoom using the result's extent — fits the map perfectly
    extent = r['extent']
    extent.update({'spatialReference': {'wkid': 4326}})
    search_map.extent = extent
    print('\n✅ Map zoomed to location. Scroll up!')
else:
    print('❌ Not found.')

## 📌 Step 5 — Draw Markers with Correct Symbol API

**Root cause of your error:**  
`draw()` internally calls `.dict()` on `symbol` and `popup`.  
A plain Python `dict` has no `.dict()` method — only pydantic models do.  

✅ Fix: use `SimpleMarkerSymbolEsriSMS(...)` for symbols, and `Geometry(...)` for shapes.

In [ ]:
draw_map = gis.map('India')
draw_map.zoom = 5
draw_map.layout.height = '500px'
draw_map

In [ ]:
# ── Define cities ─────────────────────────────────────────────────────────────
cities = [
    ('New Delhi',  28.6139,  77.2090, [220, 50,  50,  255]),   # Red
    ('Mumbai',     19.0760,  72.8777, [50,  100, 220, 255]),   # Blue
    ('Bangalore',  12.9716,  77.5946, [50,  180, 80,  255]),   # Green
    ('Hyderabad',  17.3850,  78.4867, [180, 80,  200, 255]),   # Purple
    ('Chennai',    13.0827,  80.2707, [220, 150, 50,  255]),   # Orange
]

for city_name, lat, lon, color in cities:

    # ✅ Step A: Create a Geometry object (not a raw dict)
    pt = Geometry({'x': lon, 'y': lat, 'spatialReference': {'wkid': 4326}})

    # ✅ Step B: Create a Symbol pydantic model (not a raw dict)
    symbol = SimpleMarkerSymbolEsriSMS(
        **{
            'type'   : 'esriSMS',
            'style'  : 'esriSMSCircle',   # esriSMSCircle / esriSMSSquare / esriSMSDiamond / esriSMSCross
            'color'  : color,
            'size'   : 14,
            'angle'  : 0,
            'xoffset': 0,
            'yoffset': 0,
            'outline': {
                'type' : 'esriSLS',
                'style': 'esriSLSSolid',
                'color': [255, 255, 255, 255],
                'width': 2
            }
        }
    )

    # ✅ Step C: popup must be None or omitted — plain dict causes AttributeError
    # Use 'attributes' dict for metadata instead
    draw_map.content.draw(
        pt,
        symbol=symbol,
        attributes={'name': city_name, 'lat': lat, 'lon': lon}
    )
    print(f'📍 Pinned: {city_name}')

print('\n✅ All cities pinned! Scroll up.')

## 🎯 Step 6 — All Marker Styles Side by Side

In [ ]:
style_map = gis.map('India')
style_map.zoom = 5
style_map

In [ ]:
# Different marker shapes — each placed at a different city
styles = [
    ('esriSMSCircle',   'Mumbai',    19.076,  72.878, [220, 50,  50,  255]),
    ('esriSMSSquare',   'Delhi',     28.614,  77.209, [50,  100, 220, 255]),
    ('esriSMSDiamond',  'Bangalore', 12.972,  77.595, [50,  180, 80,  255]),
    ('esriSMSCross',    'Kolkata',   22.573,  88.364, [220, 150, 50,  255]),
    ('esriSMSX',        'Chennai',   13.083,  80.271, [180, 80,  200, 255]),
]

for style, city, lat, lon, color in styles:
    pt = Geometry({'x': lon, 'y': lat, 'spatialReference': {'wkid': 4326}})
    sym = SimpleMarkerSymbolEsriSMS(**{
        'type': 'esriSMS', 'style': style, 'color': color,
        'size': 16, 'angle': 0, 'xoffset': 0, 'yoffset': 0,
        'outline': {'type': 'esriSLS', 'style': 'esriSLSSolid',
                    'color': [0, 0, 0, 200], 'width': 1}
    })
    style_map.content.draw(pt, symbol=sym)
    print(f'  {style:<22} → {city}')

print('\n✅ All 5 marker styles drawn!')

## 🖼️ Step 7 — Picture Marker (Custom Image Icon)

In [ ]:
pic_map = gis.map('India')
pic_map.zoom = 5
pic_map

In [ ]:
results = geocode('Taj Mahal, Agra', max_locations=1)
if results:
    r   = results[0]
    pt  = Geometry({'x': r['location']['x'], 'y': r['location']['y'],
                    'spatialReference': {'wkid': 4326}})

    # Use a URL-based image as the marker icon
    pic_sym = PictureMarkerSymbolEsriPMS(**{
        'type'       : 'esriPMS',
        'url'        : 'https://static.arcgis.com/images/Symbols/PeoplePlaces/esriBusinessMarker_57.png',
        'contentType': 'image/png',
        'width'      : 28,
        'height'     : 28,
        'angle'      : 0,
        'xoffset'    : 0,
        'yoffset'    : 0,
    })

    pic_map.content.draw(pt, symbol=pic_sym)
    pic_map.extent = r['extent']
    print('✅ Custom icon placed at Taj Mahal')

## 〰️ Step 8 — Draw Lines & Polygons

In [ ]:
shape_map = gis.map('India')
shape_map.zoom = 5
shape_map

In [ ]:
# ── Draw a polyline ───────────────────────────────────────────────────────────
line = Geometry({
    'paths': [[
        [77.209, 28.614],   # Delhi
        [72.878, 19.076],   # Mumbai
        [77.595, 12.972],   # Bangalore
    ]],
    'spatialReference': {'wkid': 4326}
})

line_sym = SimpleLineSymbolEsriSLS(**{
    'type' : 'esriSLS',
    'style': 'esriSLSSolid',   # esriSLSSolid / esriSLSDash / esriSLSDot / esriSLSDashDot
    'color': [255, 100, 0, 220],
    'width': 3,
})

shape_map.content.draw(line, symbol=line_sym)
print('✅ Line drawn: Delhi → Mumbai → Bangalore')

In [ ]:
# ── Draw a filled polygon ─────────────────────────────────────────────────────
polygon = Geometry({
    'rings': [[
        [68.0, 22.0],
        [88.0, 22.0],
        [88.0, 8.0],
        [68.0, 8.0],
        [68.0, 22.0],
    ]],
    'spatialReference': {'wkid': 4326}
})

fill_sym = SimpleFillSymbolEsriSFS(**{
    'type' : 'esriSFS',
    'style': 'esriSFSSolid',
    'color': [50, 130, 220, 80],    # RGBA — low alpha = transparent
    'outline': {
        'type' : 'esriSLS',
        'style': 'esriSLSSolid',
        'color': [0, 80, 180, 200],
        'width': 2
    }
})

shape_map.content.draw(polygon, symbol=fill_sym)
print('✅ Polygon drawn over southern India')

## 📦 Step 9 — Draw a FeatureSet from Geocoding

In [ ]:
fset_map = gis.map('India')
fset_map.zoom = 5
fset_map

In [ ]:
# Geocode multiple results as a single FeatureSet
india_extent = geocode('India', max_locations=1)[0]['extent']

fset = geocode(
    'University',
    search_extent=india_extent,
    max_locations=10,
    as_featureset=True           # Returns FeatureSet, not a list
)
print(f'Found: {fset}')

# Draw the entire FeatureSet in one call
sym = SimpleMarkerSymbolEsriSMS(**{
    'type': 'esriSMS', 'style': 'esriSMSDiamond',
    'color': [255, 200, 0, 255], 'size': 12,
    'angle': 0, 'xoffset': 0, 'yoffset': 0,
    'outline': {'type': 'esriSLS', 'style': 'esriSLSSolid',
                'color': [100, 80, 0, 255], 'width': 1}
})

fset_map.content.draw(fset, symbol=sym)
print('✅ FeatureSet drawn — scroll up!')

## 🌐 Step 10 — Add Public ArcGIS Online Layers

In [ ]:
layer_map = gis.map('World')
layer_map.zoom = 2
layer_map

In [ ]:
# Add a public item by its ArcGIS Online item ID
tz_item = gis.content.get('312cebfea2624e108e234220b04460b8')   # World Time Zones
layer_map.content.add(tz_item)
print('✅ World Time Zones layer added')

# List all layers on the map
print('\nLayers on map:')
for i, lyr in enumerate(layer_map.content.layers):
    print(f'  [{i}]', lyr)

In [ ]:
# Add a FeatureLayer with opacity
countries_item = gis.content.get('ac80670eb213440ea5899bbf92a04998')
countries_lyr  = countries_item.layers[0]
layer_map.content.add(countries_lyr, options={'opacity': 0.4})
print('✅ World Countries layer added at 40% opacity')

In [ ]:
# Zoom to a specific layer, then remove it
layer_map.zoom_to_layer(countries_lyr)
print('Current layers:', layer_map.content.layers)

layer_map.content.remove(0)   # Remove layer at index 0
print('After remove :', layer_map.content.layers)

## 🔄 Step 11 — Reverse Geocoding (Coordinates → Address)

In [ ]:
# Give lat/lon → get back a real address
point = {'x': 72.8777, 'y': 19.0760, 'spatialReference': {'wkid': 4326}}
result = reverse_geocode(point)

print('📍 Reverse Geocode Result:')
print(f"   Address : {result['address']['Address']}")
print(f"   City    : {result['address']['City']}")
print(f"   Country : {result['address']['CountryCode']}")
print(f"   Full    : {result['address']['Match_addr']}")

## 📋 Step 12 — Batch Geocoding (Multiple Places at Once)

In [ ]:
batch_map = gis.map('India')
batch_map.zoom = 5
batch_map

In [ ]:
landmarks = [
    'Red Fort, Delhi',
    'Charminar, Hyderabad',
    'Victoria Memorial, Kolkata',
    'Mysore Palace, Mysore',
    'Hawa Mahal, Jaipur',
]

sym = SimpleMarkerSymbolEsriSMS(**{
    'type': 'esriSMS', 'style': 'esriSMSCircle',
    'color': [255, 80, 80, 255], 'size': 12,
    'angle': 0, 'xoffset': 0, 'yoffset': 0,
    'outline': {'type': 'esriSLS', 'style': 'esriSLSSolid',
                'color': [255, 255, 255, 255], 'width': 2}
})

for place in landmarks:
    res = geocode(place, max_locations=1)
    if res:
        r  = res[0]
        pt = Geometry({'x': r['location']['x'], 'y': r['location']['y'],
                       'spatialReference': {'wkid': 4326}})
        batch_map.content.draw(pt, symbol=sym)
        print(f'  ✅ {r["address"]}')
    else:
        print(f'  ❌ Not found: {place}')

print('\n✅ All landmarks pinned!')

## 🔎 Step 13 — Search ArcGIS Online for Public Datasets

In [ ]:
results = gis.content.search('India population', item_type='Feature Layer', max_items=5)
print(f'Found {len(results)} public datasets:\n')
for item in results:
    print(f'  📦 {item.title}')
    print(f'     Type : {item.type}')
    print(f'     Owner: {item.owner}')
    print(f'     ID   : {item.id}')
    print()

## 💾 Step 14 — Export Map to HTML File

In [ ]:
# Export the current map state to a standalone HTML file
# Open it in any browser — no Python required!
batch_map.export_to_html('my_arcgis_map.html')
print('✅ Saved as my_arcgis_map.html')
print('   Open it in any browser!')

## 🌐 Step 15 — 3D Scene View

In [ ]:
from arcgis.map import Scene

scene = Scene()
scene.zoom = 4
scene

In [ ]:
# tilt: 0 = top-down, 90 = horizon view
scene.tilt    = 45
scene.heading = 0    # 0=North 90=East 180=South 270=West
print('Camera:', scene.camera)

---
## 📋 Complete Cheat Sheet

### Setup
```python
pip install arcgis arcgis-mapping
```

### Map
```python
m = gis.map('Paris')            # auto-center on place
m.zoom = 12
m.center = [lat, lon]
m.extent = geocode_result['extent']
m.layout.height = '500px'
m.basemap.basemap = 'satellite' # ✅ double .basemap.basemap
m.basemap.basemaps              # list all options
m.export_to_html('map.html')    # save to browser-viewable file
```

### Geocoding
```python
geocode('Mumbai')                          # list of results
geocode('Mumbai', max_locations=1)         # top 1 result
geocode('Mumbai', as_featureset=True)      # returns FeatureSet
reverse_geocode({'x': lon, 'y': lat, ...}) # coords → address
```

### Drawing ✅ correct pattern
```python
from arcgis.geometry import Geometry
from arcgis.map.symbols import SimpleMarkerSymbolEsriSMS

pt  = Geometry({'x': lon, 'y': lat, 'spatialReference': {'wkid': 4326}})
sym = SimpleMarkerSymbolEsriSMS(**{'type':'esriSMS','style':'esriSMSCircle',
          'color':[255,0,0,255],'size':14,'angle':0,'xoffset':0,'yoffset':0,
          'outline':{'type':'esriSLS','style':'esriSLSSolid',
                     'color':[255,255,255,255],'width':2}})
m.content.draw(pt, symbol=sym)
```

### Layers
```python
item = gis.content.get('ITEM_ID')
m.content.add(item)
m.content.add(layer, options={'opacity': 0.5})
m.content.layers          # list layers
m.content.remove(0)       # remove by index
m.zoom_to_layer(layer)    # fit map to layer
```

### Symbol types
| Class | Use for |
|---|---|
| `SimpleMarkerSymbolEsriSMS` | Points (circle, square, diamond, cross, x) |
| `SimpleLineSymbolEsriSLS` | Lines / polylines |
| `SimpleFillSymbolEsriSFS` | Filled polygons |
| `PictureMarkerSymbolEsriPMS` | Custom image icons |

### 3D
```python
from arcgis.map import Scene
s = Scene()
s.tilt = 45
s.heading = 90
```

### Save as Web Map (requires ArcGIS login)
```python
m.save({'title': 'My Map', 'tags': ['python']})
```